## ETL JSON Files

In [ ]:
import requests
import pandas as pd

### constructor-results

In [31]:
import requests
import pandas as pd

all_rows = []
for year in range(1950, 2027):
    # Get list of files in the constructor-results folder for this year
    api_url = f"https://api.github.com/repos/tamadam/f1-dataset/contents/cache/api/constructor-results/{year}"
    res = requests.get(api_url)
    if res.status_code != 200:
        print(f"Failed to fetch file list for year {year}: {res.status_code}")
        continue
    files = res.json()
    
    for file_info in files:
        if file_info['type'] == 'file' and file_info['name'].endswith('.json'):
            # Extract constructor name from filename
            filename = file_info['name']
            constructor_name = filename.replace('constructor-results-', '').replace(f'-{year}.json', '')
            
            print(filename)
            
            # Fetch the JSON data
#             url = f"https://raw.githubusercontent.com/tamadam/f1-dataset/main/cache/api/constructor-results/{year}/{filename}"
#             res_data = requests.get(url)
#             if res_data.status_code != 200:
#                 print(f"Failed to fetch data for {filename}: {res_data.status_code}")
#                 continue
#             data = res_data.json()
            
#             races = data["MRData"]["RaceTable"]["Races"]
            
#             for race in races:
#                 for result in race.get("Results", []):
#                     all_rows.append({
#                         "season": race["season"],
#                         "round": race["round"],
#                         "raceName": race["raceName"],
#                         "date": race["date"],
#                         "circuit": race["Circuit"]["circuitName"],
#                         "country": race["Circuit"]["Location"]["country"],
#                         "locality": race["Circuit"]["Location"]["locality"],
#                         "latitude": race["Circuit"]["Location"]["lat"],
#                         "longitude": race["Circuit"]["Location"]["long"],
#                         "constructorId": result["Constructor"]["constructorId"],
#                         "constructorName": result["Constructor"]["name"],
#                         "constructorNationality": result["Constructor"]["nationality"],
#                         "points": result["points"],
#                         "position": result["position"],
#                         "positionText": result["positionText"],
#                         # "wins": result["wins"],
#                     })

# # All years and constructors combined into a single DataFrame
# data_constructor_results = pd.DataFrame(all_rows)


Failed to fetch file list for year 1950: 403
Failed to fetch file list for year 1951: 403
Failed to fetch file list for year 1952: 403
Failed to fetch file list for year 1953: 403
Failed to fetch file list for year 1954: 403
Failed to fetch file list for year 1955: 403
Failed to fetch file list for year 1956: 403
Failed to fetch file list for year 1957: 403
Failed to fetch file list for year 1958: 403
Failed to fetch file list for year 1959: 403
Failed to fetch file list for year 1960: 403
Failed to fetch file list for year 1961: 403
Failed to fetch file list for year 1962: 403
Failed to fetch file list for year 1963: 403
Failed to fetch file list for year 1964: 403
Failed to fetch file list for year 1965: 403
Failed to fetch file list for year 1966: 403
Failed to fetch file list for year 1967: 403
Failed to fetch file list for year 1968: 403
Failed to fetch file list for year 1969: 403
Failed to fetch file list for year 1970: 403
Failed to fetch file list for year 1971: 403
Failed to 

### constructor-standings (X)

In [ ]:
import requests
import pandas as pd

all_rows = []
for year in range(1958, 2027):
    url = f"https://raw.githubusercontent.com/tamadam/f1-dataset/main/cache/api/constructor-standings/constructor-standings-{year}.json"
    res = requests.get(url)
    
    # Skip if file not found or any other error
    if res.status_code != 200:
        print(f"Skipping year {year}: HTTP {res.status_code}")
        continue

    data = res.json()

    standings_table = data["MRData"]["StandingsTable"]
    standings_lists = standings_table["StandingsLists"]
    
    # Iterate through each standings list (usually just one per year, but can be multiple)
    for standing_list in standings_lists:
        season = standing_list["season"]
        round_num = standing_list["round"]
        
        # Iterate through all constructors in this standings list
        for constructor_standing in standing_list["ConstructorStandings"]:
            all_rows.append({
                "season": season,
                "round": round_num,
                "position": constructor_standing.get("position", constructor_standing.get("positionText", "")),
                "positionText": constructor_standing.get("positionText", ""),
                "points": constructor_standing["points"],
                "wins": constructor_standing["wins"],
                "constructorId": constructor_standing["Constructor"]["constructorId"],
                "constructorName": constructor_standing["Constructor"]["name"],
                "constructorNationality": constructor_standing["Constructor"]["nationality"],
            })

# All years combined into a single DataFrame
constructor_data_standings = pd.DataFrame(all_rows)

Skipping year 2026: HTTP 404


### constructors (X)

In [81]:
import requests
import pandas as pd

all_rows = []

for i in range(1950, 2026):
    url = f"https://raw.githubusercontent.com/tamadam/f1-dataset/main/cache/api/constructors/constructors-{i}.json"
    
    try:
        # Request data
        res = requests.get(url)
        res.raise_for_status()  # handle HTTP error (404, dll)

        data = res.json()

        # Ambil struktur JSON
        constructor_table = data["MRData"]["ConstructorTable"]
        season = constructor_table["season"]
        constructors = constructor_table["Constructors"]

        # Loop constructor
        for constructor in constructors:
            try:
                all_rows.append({
                    "season": season,
                    "constructorId": constructor.get("constructorId"),
                    "name": constructor.get("name"),
                    "nationality": constructor.get("nationality")
                })
            except Exception as e:
                print(f"Error parsing constructor in season {i}: {e}")
                continue

    except requests.exceptions.HTTPError:
        print(f"Skip {i} - File not found (HTTP Error)")
        continue

    except KeyError as e:
        print(f"Skip {i} - JSON structure issue: {e}")
        continue

    except Exception as e:
        print(f"Unexpected error at {i}: {e}")
        continue

# Convert ke DataFrame
data_constructor = pd.DataFrame(all_rows)

In [82]:
data_constructor

,season,constructorId,name,nationality
0,1950,adams,Adams,American
1,1950,alfa,Alfa Romeo,Swiss
2,1950,alta,Alta,British
3,1950,cooper,Cooper,British
4,1950,deidt,Deidt,American
...,...,...,...,...
1116,2025,mercedes,Mercedes,German
1117,2025,rb,RB F1 Team,Italian
1118,2025,red_bull,Red Bull,Austrian
1119,2025,sauber,Sauber,Swiss


### drivers (X)

In [77]:
import requests
import pandas as pd

all_rows = []

for i in range(1950, 2026):
    url = f"https://raw.githubusercontent.com/tamadam/f1-dataset/main/cache/api/drivers/drivers-{i}.json"
    
    try:
        # Request data
        res = requests.get(url)
        res.raise_for_status()  # handle HTTP error (404, dll)

        data = res.json()

        # Ambil struktur JSON
        driver_table = data["MRData"]["DriverTable"]
        season = driver_table["season"]
        drivers = driver_table["Drivers"]

        # Loop driver
        for driver in drivers:
            try:
                all_rows.append({
                    "season": season,
                    "driverId": driver.get("driverId"),
                    "driverFullname": f"{driver.get('givenName', '')} {driver.get('familyName', '')}",
                    "dateOfBirth": driver.get("dateOfBirth"),
                    "nationality": driver.get("nationality")
                })
            except Exception as e:
                print(f"Error parsing driver in season {i}: {e}")
                continue

    except requests.exceptions.HTTPError:
        print(f"Skip {i} - File not found (HTTP Error)")
        continue

    except KeyError as e:
        print(f"Skip {i} - JSON structure issue: {e}")
        continue

    except Exception as e:
        print(f"Unexpected error at {i}: {e}")
        continue

# Convert ke DataFrame
data_driver = pd.DataFrame(all_rows)

In [ ]:
data_driver

,season,driverId,driverFullname,dateOfBirth,nationality
0,1950,ader,Walt Ader,1913-12-15,American
1,1950,agabashian,Fred Agabashian,1913-08-21,American
2,1950,ascari,Alberto Ascari,1918-07-13,Italian
3,1950,banks,Henry Banks,1913-06-14,American
4,1950,bettenhausen,Tony Bettenhausen,1916-09-12,American
...,...,...,...,...,...
3224,2025,sainz,Carlos Sainz,1994-09-01,Spanish
3225,2025,cian_shields,Cian Shields,None,None
3226,2025,stroll,Lance Stroll,1998-10-29,Canadian
3227,2025,tsunoda,Yuki Tsunoda,2000-05-11,Japanese


### driver-standings (X)

In [108]:
import requests
import pandas as pd

all_rows = []

for year in range(1950, 2026):

    url = f"https://raw.githubusercontent.com/tamadam/f1-dataset/main/cache/api/driver-standings/driver-standings-{year}.json"

    try:
        res = requests.get(url)
        
        if res.status_code != 200:
            print(f"Skip {year}")
            continue

        data = res.json()

        standings_table = data.get("MRData", {}).get("StandingsTable", {})
        season = standings_table.get("season")
        round_ = standings_table.get("round")
        standings_lists = standings_table.get("StandingsLists", [])

        for standing in standings_lists:
            for driver_standing in standing.get("DriverStandings", []):

                driver = driver_standing.get("Driver", {})
                constructors = driver_standing.get("Constructors", [])

                # Driver Info
                driverId = driver.get("driverId")
                driverFullName = f"{driver.get('givenName', '')} {driver.get('familyName', '')}".strip()
                driverDOB = driver.get("dateOfBirth")
                driverNationality = driver.get("nationality")

                # Standing Info
                driverPosition = driver_standing.get("position")
                driverPoints = driver_standing.get("points")
                driverWins = driver_standing.get("wins")

                for constructor in constructors:
                    all_rows.append({
                        "season": season,
                        "round": round_,

                        "driverPosition": driverPosition,
                        "driverPoints": driverPoints,
                        "driverWins": driverWins,

                        "driverId": driverId,
                        "driverFullName": driverFullName,
                        "driverDateOfBirth": driverDOB,
                        "driverNationality": driverNationality,

                        "constructorId": constructor.get("constructorId"),
                        "constructorName": constructor.get("name"),
                        "constructorNationality": constructor.get("nationality"),
                    })

    except Exception as e:
        print(f"Error {year}: {e}")
        continue

# Convert ke DataFrame
df_driver_standings = pd.DataFrame(all_rows)

### driver-results (X)

In [ ]:
import requests
import pandas as pd

all_rows = []

# ambil dari dataframe driver kamu
driver_season_list = data_driver[["season", "driverId"]].drop_duplicates()

for _, row in driver_season_list.iterrows():
    season = row["season"]
    driver_id = row["driverId"]

    url = f"https://raw.githubusercontent.com/tamadam/f1-dataset/main/cache/api/driver-results/{season}/driver-result-{driver_id}-{season}.json"  

    try:
        res = requests.get(url)

        if res.status_code != 200:
            continue

        data = res.json()

        race_table = data["MRData"]["RaceTable"]
        races = race_table.get("Races", [])

        for race in races:
            circuit = race.get("Circuit", {})
            location = circuit.get("Location", {})

            for result in race.get("Results", []):
                driver = result.get("Driver", {})
                constructor = result.get("Constructor", {})

                all_rows.append({
                    "season": season,
                    "driverId": driver_id,
                    "round": race.get("round"),
                    "raceName": race.get("raceName"),
                    "circuitId": circuit.get("circuitId"),
                    "circuitName": circuit.get("circuitName"),
                    "lat": location.get("lat"),
                    "long": location.get("long"),
                    "locality": location.get("locality"),
                    "country": location.get("country"),
                    "date": race.get("date"),
                    "number": result.get("number"),
                    "position": result.get("position"),
                    "points": result.get("points"),
                    "givenName": driver.get("givenName"),
                    "familyName": driver.get("familyName"),
                    "dateOfBirth": driver.get("dateOfBirth"),
                    "nationality": driver.get("nationality"),
                    "constructorId": constructor.get("constructorId"),
                    "grid": result.get("grid"),
                    "laps": result.get("laps"),
                    "status": result.get("status")
                })

    except Exception as e:
        print(f"Error {season}-{driver_id}: {e}")
        continue

df_results = pd.DataFrame(all_rows)

Empty DataFrame
Columns: []
Index: []

Total rows: 0


### fastest-laps (X)

In [104]:
import requests
import pandas as pd

all_rows = []

for year in range(1950, 2026):

    url = f"https://raw.githubusercontent.com/tamadam/f1-dataset/main/cache/api/fastest-laps/fastest-laps-{year}.json"

    try:
        res = requests.get(url)
        res.raise_for_status()

        data = res.json()

        race_table = data["MRData"]["RaceTable"]
        season = race_table.get("season")
        races = race_table.get("Races", [])

        for race in races:
            try:
                round_ = race.get("round")
                raceName = race.get("raceName")
                date = race.get("date")

                circuit = race.get("Circuit", {})
                location = circuit.get("Location", {})

                circuitId = circuit.get("circuitId")
                circuitName = circuit.get("circuitName")
                lat = location.get("lat")
                long_ = location.get("long")
                locality = location.get("locality")
                country = location.get("country")

                for result in race.get("Results", []):
                    try:
                        driver = result.get("Driver", {})
                        constructor = result.get("Constructor", {})
                        time_data = result.get("Time", {})
                        fastest = result.get("FastestLap", {})
                        fastest_time = fastest.get("Time", {})
                        fastest_speed = fastest.get("AverageSpeed", {})

                        all_rows.append({
                            # Race
                            "season": season,
                            "round": round_,
                            "raceName": raceName,
                            "date": date,

                            # Circuit
                            "circuitId": circuitId,
                            "circuitName": circuitName,
                            "lat": lat,
                            "long": long_,
                            "locality": locality,
                            "country": country,

                            # Driver
                            "driverId": driver.get("driverId"),
                            "givenName": driver.get("givenName"),
                            "familyName": driver.get("familyName"),
                            "dateOfBirth": driver.get("dateOfBirth"),
                            "nationality": driver.get("nationality"),

                            # Result
                            "constructorId": constructor.get("constructorId"),
                            "grid": result.get("grid"),
                            "position": result.get("position"),
                            "points": result.get("points"),
                            "laps": result.get("laps"),
                            "status": result.get("status"),

                            # Race Time
                            "raceTime": time_data.get("time"),
                            "raceMillis": time_data.get("millis"),

                            # Fastest Lap
                            "fastestLapRank": fastest.get("rank"),
                            "fastestLapLap": fastest.get("lap"),
                            "fastestLapTime": fastest_time.get("time"),
                            "fastestLapSpeed": fastest_speed.get("speed"),
                            "fastestLapSpeedUnit": fastest_speed.get("units")
                        })

                    except Exception as e:
                        print(f"Error parsing result: {e}")
                        continue

            except Exception as e:
                print(f"Error parsing race: {e}")
                continue

    except Exception as e:
        print(f"Error request: {e}")

# Convert ke DataFrame
df_fastest = pd.DataFrame(all_rows)

### qualifying-results (X)

In [116]:
import requests
import pandas as pd

all_rows = []

for year in range(2006, 2026):
    try:
        for round_ in range(1, 22):
            
            url = f"https://raw.githubusercontent.com/tamadam/f1-dataset/main/cache/api/qualifying-results/{year}/qualifying-result-{round_}-{year}.json"

            try:
                res = requests.get(url)
                res.raise_for_status()

                data = res.json()

                race_table = data["MRData"]["RaceTable"]
                season = race_table.get("season")
                round__ = race_table.get("round")
                races = race_table.get("Races", [])

                for race in races:
                    try:
                        raceName = race.get("raceName")
                        date = race.get("date")
                        time_ = race.get("time")

                        circuit = race.get("Circuit", {})
                        location = circuit.get("Location", {})

                        circuitId = circuit.get("circuitId")
                        circuitName = circuit.get("circuitName")
                        lat = location.get("lat")
                        long_ = location.get("long")
                        locality = location.get("locality")
                        country = location.get("country")

                        for qualifyingResult in race.get("QualifyingResults", []):
                            try:
                                driver = qualifyingResult.get("Driver", {})
                                constructor = qualifyingResult.get("Constructor", {})

                                all_rows.append({
                                    # Race
                                    "season": season,
                                    "round": round__,
                                    "raceName": raceName,
                                    "date": date,
                                    "time":time_,
        
                                    # Circuit
                                    "circuitId": circuitId,
                                    "circuitName": circuitName,
                                    "lat": lat,
                                    "long": long_,
                                    "locality": locality,
                                    "country": country,

                                    # Driver
                                    "driverId": driver.get("driverId"),
                                    "driverCode": driver.get("code"),
                                    "fullName": f"{driver.get('givenName')} {driver.get('familyName')}",
                                    "dateOfBirth": driver.get("dateOfBirth"),
                                    "nationality": driver.get("nationality"),

                                    # Result
                                    "constructorId": constructor.get("constructorId"),
                                    "name": result.get("name"),
                                    "nationality": result.get("nationality"),

                                    # Race Time
                                    "Q1": qualifyingResult.get("Q1"),
                                    "Q2": qualifyingResult.get("Q2"),
                                    "Q3": qualifyingResult.get("Q3")
                                })

                            except Exception as e:
                                print(f"Error parsing result: {e}")
                                continue

                    except Exception as e:
                        print(f"Error parsing race: {e}")
                        continue

            except Exception as e:
                print(f"Error request: {e}")
                continue
            
    except Exception as e:
        print(f"Error round: {e}")
            

# Convert ke DataFrame
df_qualifying_result = pd.DataFrame(all_rows)

Error request: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/tamadam/f1-dataset/main/cache/api/qualifying-results/2006/qualifying-result-19-2006.json
Error request: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/tamadam/f1-dataset/main/cache/api/qualifying-results/2006/qualifying-result-20-2006.json
Error request: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/tamadam/f1-dataset/main/cache/api/qualifying-results/2006/qualifying-result-21-2006.json
Error request: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/tamadam/f1-dataset/main/cache/api/qualifying-results/2007/qualifying-result-18-2007.json
Error request: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/tamadam/f1-dataset/main/cache/api/qualifying-results/2007/qualifying-result-19-2007.json
Error request: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/tamadam/f1-dataset/main/cache/api/quali

### race-results

In [ ]:
import requests
import pandas as pd

all_rows = []

for year in range(1950, 2026):
    try:
        for round_ in range(1, 30):
            
            url = f"https://raw.githubusercontent.com/tamadam/f1-dataset/main/cache/api/race-results/{year}/race-result-{round_}-{year}.json"

            try:
                res = requests.get(url)
                res.raise_for_status()

                data = res.json()

                race_table = data["MRData"]["RaceTable"]
                season = race_table.get("season")
                round__ = race_table.get("round")
                races = race_table.get("Races", [])

                for race in races:
                    try:
                        raceName = race.get("raceName")
                        date = race.get("date")
                        # time_ = race.get("time")

                        circuit = race.get("Circuit", {})
                        location = circuit.get("Location", {})

                        circuitId = circuit.get("circuitId")
                        circuitName = circuit.get("circuitName")
                        lat = location.get("lat")
                        long_ = location.get("long")
                        locality = location.get("locality")
                        country = location.get("country")

                        for result in race.get("Results", []):
                            try:
                                driver = result.get("Driver", {})
                                constructor = result.get("Constructor", {})
                                time_ = result.get("Time", {})

                                all_rows.append({
                                    # Race
                                    "season": season,
                                    "round": round__,
                                    "raceName": raceName,
                                    "date": date,
                                    # "time":time_,
        
                                    # Circuit
                                    "circuitId": circuitId,
                                    "circuitName": circuitName,
                                    "lat": lat,
                                    "long": long_,
                                    "locality": locality,
                                    "country": country,

                                    # Driver Result
                                    "driverNumber": result.get("number"),
                                    "driverPosition": result.get("position"),
                                    "driverPoints": result.get("points"),
                                    
                                    # Driver
                                    "driverId": driver.get("driverId"),
                                    "driverFullName": f"{driver.get('givenName')} {driver.get('familyName')}",
                                    "driverDateOfBirth": driver.get("dateOfBirth"),
                                    "driverNationality": driver.get("nationality"),
                                    
                                    # Constructor
                                    "constructorId": constructor.get("constructorId"),
                                    "constructorName": constructor.get("name"),
                                    "constructorNationality": constructor.get("nationality"),

                                    # Race Time
                                    "grid": result.get("grid"),
                                    "laps": result.get("laps"),
                                    "status": result.get("status"),
                                    "timeResult": time_.get("time"),
                                    "millisTimeResult": time_.get("millis"),
                                })

                            except Exception as e:
                                print(f"Error parsing result: {e}")
                                continue

                    except Exception as e:
                        print(f"Error parsing race: {e}")
                        continue

            except Exception as e:
                print(f"Error request: {e}")
                continue

    except Exception as e:
        print(f"Error round: {e}")


# Convert ke DataFrame
df_race_results = pd.DataFrame(all_rows)

Error request: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/tamadam/f1-dataset/main/cache/api/race-results/1950/race-result-8-1950.json
Error request: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/tamadam/f1-dataset/main/cache/api/race-results/1950/race-result-9-1950.json
Error request: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/tamadam/f1-dataset/main/cache/api/race-results/1950/race-result-10-1950.json
Error request: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/tamadam/f1-dataset/main/cache/api/race-results/1950/race-result-11-1950.json
Error request: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/tamadam/f1-dataset/main/cache/api/race-results/1950/race-result-12-1950.json
Error request: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/tamadam/f1-dataset/main/cache/api/race-results/1950/race-result-13-1950.json
Error request: 404 Clien

In [118]:
df_race_results

,season,round,raceName,date,time,circuitId,circuitName,lat,long,locality,...,driverId,driverFullName,driverDateOfBirth,driverNationality,constructorId,constructorName,constructorNationality,grid,laps,status
0,1950,1,British Grand Prix,1950-05-13,None,silverstone,Silverstone Circuit,52.0786,-1.01694,Silverstone,...,farina,Nino Farina,1906-10-30,Italian,alfa,Alfa Romeo,Swiss,1,70,Finished
1,1950,1,British Grand Prix,1950-05-13,None,silverstone,Silverstone Circuit,52.0786,-1.01694,Silverstone,...,fagioli,Luigi Fagioli,1898-06-09,Italian,alfa,Alfa Romeo,Swiss,2,70,Finished
2,1950,1,British Grand Prix,1950-05-13,None,silverstone,Silverstone Circuit,52.0786,-1.01694,Silverstone,...,reg_parnell,Reg Parnell,1911-07-02,British,alfa,Alfa Romeo,Swiss,4,70,Finished
3,1950,1,British Grand Prix,1950-05-13,None,silverstone,Silverstone Circuit,52.0786,-1.01694,Silverstone,...,cabantous,Yves Cabantous,1904-10-08,French,lago,Talbot-Lago,French,6,68,+2 Laps
4,1950,1,British Grand Prix,1950-05-13,None,silverstone,Silverstone Circuit,52.0786,-1.01694,Silverstone,...,rosier,Louis Rosier,1905-11-05,French,lago,Talbot-Lago,French,9,68,+2 Laps
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25759,2025,24,Abu Dhabi Grand Prix,2025-12-07,13:00:00Z,yas_marina,Yas Marina Circuit,24.4672,54.6031,Abu Dhabi,...,albon,Alexander Albon,1996-03-23,Thai,williams,Williams,British,17,58,Finished
25760,2025,24,Abu Dhabi Grand Prix,2025-12-07,13:00:00Z,yas_marina,Yas Marina Circuit,24.4672,54.6031,Abu Dhabi,...,hadjar,Isack Hadjar,2004-09-28,French,rb,RB F1 Team,Italian,9,57,Lapped
25761,2025,24,Abu Dhabi Grand Prix,2025-12-07,13:00:00Z,yas_marina,Yas Marina Circuit,24.4672,54.6031,Abu Dhabi,...,lawson,Liam Lawson,2002-02-11,New Zealander,rb,RB F1 Team,Italian,13,57,Lapped
25762,2025,24,Abu Dhabi Grand Prix,2025-12-07,13:00:00Z,yas_marina,Yas Marina Circuit,24.4672,54.6031,Abu Dhabi,...,gasly,Pierre Gasly,1996-02-07,French,alpine,Alpine F1 Team,French,19,57,Lapped


### races (X)

In [6]:
import requests
import pandas as pd

all_rows = []
for i in range(1950, 2027):
    url = f"https://raw.githubusercontent.com/tamadam/f1-dataset/main/cache/api/races/races-{i}.json"
    res = requests.get(url)
    res.raise_for_status()
    data = res.json()

    races = data["MRData"]["RaceTable"]["Races"]

    for race in races:
        all_rows.append({
            "season": race["season"],
            "round": race["round"],
            "raceName": race["raceName"],
            "date": race["date"],
            "circuit": race["Circuit"]["circuitName"],
            "country": race["Circuit"]["Location"]["country"],
            "locality": race["Circuit"]["Location"]["locality"],
            "latitude": race["Circuit"]["Location"]["lat"],
            "longitude": race["Circuit"]["Location"]["long"],
        })

# All years combined into a single DataFrame
data_races = pd.DataFrame(all_rows)


In [101]:
data_races

,season,round,raceName,date,circuit,country,locality,latitude,longitude
0,1950,1,British Grand Prix,1950-05-13,Silverstone Circuit,UK,Silverstone,52.0786,-1.01694
1,1950,2,Monaco Grand Prix,1950-05-21,Circuit de Monaco,Monaco,Monte-Carlo,43.7347,7.42056
2,1950,3,Indianapolis 500,1950-05-30,Indianapolis Motor Speedway,USA,Indianapolis,39.795,-86.2347
3,1950,4,Swiss Grand Prix,1950-06-04,Circuit Bremgarten,Switzerland,Bern,46.9589,7.40194
4,1950,5,Belgian Grand Prix,1950-06-18,Circuit de Spa-Francorchamps,Belgium,Spa,50.4372,5.97139
...,...,...,...,...,...,...,...,...,...
1168,2026,20,Mexico City Grand Prix,2026-11-01,Autódromo Hermanos Rodríguez,Mexico,Mexico City,19.4042,-99.0907
1169,2026,21,Brazilian Grand Prix,2026-11-08,Autódromo José Carlos Pace,Brazil,São Paulo,-23.7036,-46.6997
1170,2026,22,Las Vegas Grand Prix,2026-11-22,Las Vegas Strip Street Circuit,USA,Las Vegas,36.1147,-115.173
1171,2026,23,Qatar Grand Prix,2026-11-29,Losail International Circuit,Qatar,Lusail,25.49,51.4542


### sprint-results

In [ ]:
import requests
import pandas as pd

all_rows = []

for year in range(2021, 2026):
    try:
        for round_ in range(1, 30):
            
            url = f"https://raw.githubusercontent.com/tamadam/f1-dataset/main/cache/api/sprint-results/{year}/sprint-result-{round_}-{year}.json"

            try:
                res = requests.get(url)
                res.raise_for_status()

                data = res.json()

                race_table = data["MRData"]["RaceTable"]
                season = race_table.get("season")
                round_race = race_table.get("round")
                races = race_table.get("Races", [])

                for race in races:
                    try:
                        raceName = race.get("raceName")
                        date = race.get("date")
                        race_time = race.get("time")

                        circuit = race.get("Circuit", {})
                        location = circuit.get("Location", {})

                        circuitId = circuit.get("circuitId")
                        circuitName = circuit.get("circuitName")
                        lat = location.get("lat")
                        long_ = location.get("long")
                        locality = location.get("locality")
                        country = location.get("country")

                        for sprint_result in race.get("SprintResults", []):
                            try:
                                driver = sprint_result.get("Driver", {})
                                constructor = sprint_result.get("Constructor", {})
                                sprint_result_time = sprint_result.get("Time", {})
                                fastest_lap_dir = sprint_result.get("FastestLap", {})
                                fastest_lap_time = fastest_lap_dir.get("Time", {})
                                fastest_lap_time_dir = fastest_lap_dir.get("Time", {})
                                fastest_lap = fastest_lap_dir.get("lap")

                                all_rows.append({
                                    # Race
                                    "season": season,
                                    "round": round_race,
                                    "raceName": raceName,
                                    "date": date,
                                    "time": race_time,

                                    # Circuit
                                    "circuitId": circuitId,
                                    "circuitName": circuitName,
                                    "lat": lat,
                                    "long": long_,
                                    "locality": locality,
                                    "country": country,

                                    # Sprint Result
                                    "sprintResultNumber": sprint_result.get("number"),
                                    "sprintResultPosition": sprint_result.get("position"),
                                    "sprintResultPoints": sprint_result.get("points"),
                                    
                                    # Driver
                                    "driverId": driver.get("driverId"),
                                    "driverFullName": f"{driver.get('givenName')} {driver.get('familyName')}",
                                    "driverDateOfBirth": driver.get("dateOfBirth"),
                                    "driverNationality": driver.get("nationality"),
                                    
                                    # Constructor
                                    "constructorId": constructor.get("constructorId"),
                                    "constructorName": constructor.get("name"),
                                    "constructorNationality": constructor.get("nationality"),

                                    # Sprint Result Time
                                    "grid": sprint_result.get("grid"),
                                    "laps": sprint_result.get("laps"),
                                    "status": sprint_result.get("status"),

                                    # Sprint Result Time (Fastest Lap)
                                    "fastestLapTime": fastest_lap_time.get("Time", {}).get("time"),
                                    "fastestLap": fastest_lap
                                })

                            except Exception as e:
                                print(f"Error parsing result: {e}")
                                continue

                    except Exception as e:
                        print(f"Error parsing race: {e}")
                        continue

            except Exception as e:
                print(f"Error request: {e}")
                continue
            
    except Exception as e:
        print(f"Error round: {e}")
            

# Convert ke DataFrame
df_sprint_results = pd.DataFrame(all_rows)

Error request: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/tamadam/f1-dataset/main/cache/api/sprint-results/2021/sprint-result-1-2021.json
Error request: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/tamadam/f1-dataset/main/cache/api/sprint-results/2021/sprint-result-2-2021.json
Error request: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/tamadam/f1-dataset/main/cache/api/sprint-results/2021/sprint-result-3-2021.json
Error request: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/tamadam/f1-dataset/main/cache/api/sprint-results/2021/sprint-result-4-2021.json
Error request: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/tamadam/f1-dataset/main/cache/api/sprint-results/2021/sprint-result-5-2021.json
Error request: 404 Client Error: Not Found for url: https://raw.githubusercontent.com/tamadam/f1-dataset/main/cache/api/sprint-results/2021/sprint-result-6-2021.json
Erro

In [121]:
df_sprint_results

,season,round,raceName,date,time,circuitId,circuitName,lat,long,locality,...,driverDateOfBirth,driverNationality,constructorId,constructorName,constructorNationality,grid,laps,status,fastestLapTime,fastestLap
0,2021,10,British Grand Prix,2021-07-18,14:00:00Z,silverstone,Silverstone Circuit,52.0786,-1.01694,Silverstone,...,1997-09-30,Dutch,red_bull,Red Bull,Austrian,2,17,Finished,None,14
1,2021,10,British Grand Prix,2021-07-18,14:00:00Z,silverstone,Silverstone Circuit,52.0786,-1.01694,Silverstone,...,1985-01-07,British,mercedes,Mercedes,German,1,17,Finished,None,17
2,2021,10,British Grand Prix,2021-07-18,14:00:00Z,silverstone,Silverstone Circuit,52.0786,-1.01694,Silverstone,...,1989-08-28,Finnish,mercedes,Mercedes,German,3,17,Finished,None,17
3,2021,10,British Grand Prix,2021-07-18,14:00:00Z,silverstone,Silverstone Circuit,52.0786,-1.01694,Silverstone,...,1997-10-16,Monegasque,ferrari,Ferrari,Italian,4,17,Finished,None,16
4,2021,10,British Grand Prix,2021-07-18,14:00:00Z,silverstone,Silverstone Circuit,52.0786,-1.01694,Silverstone,...,1999-11-13,British,mclaren,McLaren,British,6,17,Finished,None,16
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
475,2025,23,Qatar Grand Prix,2025-11-30,16:00:00Z,losail,Losail International Circuit,25.49,51.4542,Lusail,...,1987-08-19,German,sauber,Sauber,Swiss,14,19,Finished,None,5
476,2025,23,Qatar Grand Prix,2025-11-30,16:00:00Z,losail,Losail International Circuit,25.49,51.4542,Lusail,...,1985-01-07,British,ferrari,Ferrari,Italian,18,19,Finished,None,8
477,2025,23,Qatar Grand Prix,2025-11-30,16:00:00Z,losail,Losail International Circuit,25.49,51.4542,Lusail,...,1996-02-07,French,alpine,Alpine F1 Team,French,19,19,Finished,None,19
478,2025,23,Qatar Grand Prix,2025-11-30,16:00:00Z,losail,Losail International Circuit,25.49,51.4542,Lusail,...,1998-10-29,Canadian,aston_martin,Aston Martin,British,17,19,Finished,None,15


In [10]:
root_data = data['MRData']

In [12]:
root_data['RaceTable']['Races']

[{'season': '1950',
  'round': '1',
  'url': 'https://en.wikipedia.org/wiki/1950_British_Grand_Prix',
  'raceName': 'British Grand Prix',
  'Circuit': {'circuitId': 'silverstone',
   'url': 'https://en.wikipedia.org/wiki/Silverstone_Circuit',
   'circuitName': 'Silverstone Circuit',
   'Location': {'lat': '52.0786',
    'long': '-1.01694',
    'locality': 'Silverstone',
    'country': 'UK'}},
  'date': '1950-05-13'},
 {'season': '1950',
  'round': '2',
  'url': 'https://en.wikipedia.org/wiki/1950_Monaco_Grand_Prix',
  'raceName': 'Monaco Grand Prix',
  'Circuit': {'circuitId': 'monaco',
   'url': 'https://en.wikipedia.org/wiki/Circuit_de_Monaco',
   'circuitName': 'Circuit de Monaco',
   'Location': {'lat': '43.7347',
    'long': '7.42056',
    'locality': 'Monte-Carlo',
    'country': 'Monaco'}},
  'date': '1950-05-21'},
 {'season': '1950',
  'round': '3',
  'url': 'https://en.wikipedia.org/wiki/1950_Indianapolis_500',
  'raceName': 'Indianapolis 500',
  'Circuit': {'circuitId': 'indi

In [53]:
import json
import pandas as pd

with open("data.json") as f:
    data = json.load(f)

races = data["MRData"]["RaceTable"]["Races"]

rows = []

for race in races:
    rows.append({
        "season": race["season"],
        "round": race["round"],
        "raceName": race["raceName"],
        "date": race["date"],
        "circuit": race["Circuit"]["circuitName"],
        "country": race["Circuit"]["Location"]["country"],
        "locality": race["Circuit"]["Location"]["locality"],
        "latitude": [race["Circuit"]["Location"]["lat"]][0],
        "longitude": [race["Circuit"]["Location"]["long"]][0],
    })

df = pd.DataFrame(rows)

In [54]:
df

,season,round,raceName,date,circuit,country,locality,latitude,longitude
0,1950,1,British Grand Prix,1950-05-13,Silverstone Circuit,UK,Silverstone,52.0786,-1.01694
1,1950,2,Monaco Grand Prix,1950-05-21,Circuit de Monaco,Monaco,Monte-Carlo,43.7347,7.42056
2,1950,3,Indianapolis 500,1950-05-30,Indianapolis Motor Speedway,USA,Indianapolis,39.795,-86.2347
3,1950,4,Swiss Grand Prix,1950-06-04,Circuit Bremgarten,Switzerland,Bern,46.9589,7.40194
4,1950,5,Belgian Grand Prix,1950-06-18,Circuit de Spa-Francorchamps,Belgium,Spa,50.4372,5.97139
5,1950,6,French Grand Prix,1950-07-02,Reims-Gueux,France,Reims,49.2542,3.93083
6,1950,7,Italian Grand Prix,1950-09-03,Autodromo Nazionale di Monza,Italy,Monza,45.6156,9.28111


In [56]:
df["date"] = pd.to_datetime(df["date"])

In [ ]:
df

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   season     7 non-null      object        
 1   round      7 non-null      object        
 2   raceName   7 non-null      object        
 3   date       7 non-null      datetime64[ns]
 4   circuit    7 non-null      object        
 5   country    7 non-null      object        
 6   locality   7 non-null      object        
 7   latitude   7 non-null      object        
 8   longitude  7 non-null      object        
dtypes: datetime64[ns](1), object(8)
memory usage: 636.0+ bytes
